# Nearest Neighbour Fingerprint Evaluation

This notebook aggregates and evaluates the predictions generated by the nearest-neighbour baseline.

Specifically, we load cached prediction files containing:

- the **ground-truth molecular fingerprints**
- the **predicted fingerprints** obtained from nearest-neighbour retrieval

For each dataset split, we compute the **Jaccard similarity** between the predicted and ground-truth fingerprints.  
The Jaccard index measures the overlap between two binary fingerprints and is defined as:

$$J(A,B) = \frac{|A \cap B|}{|A \cup B|}$$

where \(A\) and \(B\) are the predicted and ground-truth fingerprint vectors.

The notebook iterates through all stored result files, computes the **mean Jaccard similarity**, and prints the results for each experiment.

This provides a simple and interpretable metric for evaluating how well the nearest-neighbour baseline recovers the correct molecular fingerprint.

Import libraries

In [4]:
import os
import numpy as np
from pathlib import Path

from utils import load_pickle, load_json

Settings

In [25]:
results_folder = Path("../data/processed_data")
splits_folder = Path("../data/splits/")

NPLIB1 = load_pickle(results_folder / "NPLIB1.pkl")
MSG = load_pickle(results_folder / "massspecgym.pkl")

NPLIB1 = {str(r["id_"]): r for r in NPLIB1}
MSG = {str(r["id_"]): r for r in MSG}

Calculate the overlap

In [ ]:
for chosen_data in ["NPLIB1", "massspecgym"]:

    if chosen_data == "NPLIB1": 
        data = NPLIB1
        key = "inchikey_original"
    elif chosen_data == "massspecgym": 
        data = MSG 
        key = "inchikey"

    split = load_json(splits_folder / chosen_data / f"scaffold.json")
    train_id, test_id = split["train"], split["test"]
    train_id = [i.replace(".pkl", "") for i in train_id]
    test_id = [i.replace(".pkl", "") for i in test_id]

    train_inchikey = [data[i][key] for i in train_id]
    test_inchikey = [data[i][key] for i in test_id]

    overlap = [i for i in test_inchikey if i in train_inchikey]

    print(chosen_data)
    print(len(overlap) / len(test_inchikey) * 100)
    print(len(train_id), len(test_id))


Helper Functions

In [ ]:
def batch_jaccard_index(FP_pred, FP):

    # Intersection = bitwise AND
    intersection = np.logical_and(FP, FP_pred).sum(axis=1)

    # Union = bitwise OR
    union = np.logical_or(FP, FP_pred).sum(axis=1)

    # Avoid division-by-zero by adding a small epsilon
    jaccard_scores = intersection / (union + 1e-9)

    return jaccard_scores

Get the results now

In [ ]:
for f in os.listdir(results_folder):

    _, _, test_FP, pred_FP = load_pickle(results_folder / f)
    j = round(batch_jaccard_index(pred_FP, test_FP).mean(),3)

    print(f, j)